# Fixed-layer ablation on Kaggle (GPU T4, internet on)

The pre-registered plan, including Amendment 1, is in `docs/ablation_plan.md`. For each layer in
`LAYERS` the notebook:

1. Extracts that one layer's wav2vec2 embeddings on the GPU from the frozen distorted wavs.
   Only the 2,996 severe clips are needed (train, val and test). They go to local disk, never to
   the output.
2. Trains the tempo mapper with per-layer input normalisation (statistics from the train split
   only). The recipe is identical for every layer: `configs/train.yaml` at the pinned `REF`
   (batch 4, Adam lr 1e-3, ReduceLROnPlateau, 50 epochs, seed 42, severity severe).
3. Writes `best.pt`, `run_info.json`, `input_norm.npz`, `train.log` and `loss_curve.png` to
   `/kaggle/working/runs/cv27_layer<NN>_tempo/`.
4. Deletes the embeddings and the temporary checkpoints before the next layer.

**Layer index.** A layer is the index into `outputs.hidden_states`: 0 is the feature projection,
k is transformer block k, and 24 includes the final LayerNorm.

**Before training,** the notebook checks its inputs and stops on any mismatch:
- Every input file must match its checksum.
- The four manifests must equal the repo's.
- Layer 9, freshly extracted for 12 test clips, must match the W2 archive (relative L2 error at most 1e-3).

**Running unattended.** It works as a background **Save & Run All (Commit)**. There are no inputs
and no Google Drive.
- A time guard skips the remaining layers rather than run past `TIME_BUDGET_H`.
- A layer that already has a `DONE` marker in the output is skipped.

**Needs:** the Kaggle Dataset `thai-dsr-cv27-ablation` attached, a GPU T4 accelerator and
internet on (to clone the repo and download the wav2vec2 model).

In [ ]:
# ---- settings -------------------------------------------------------------
LAYERS = [3, 6, 9, 12, 18, 24]   # hidden_states indices; any subset, run in this order
REF = "b6c04e315f407134e9919fae77eea88dbbaea2c7"                  # pinned commit of github.com/KankaveeRamsri/thai-dsr (code + configs)
REPO_URL = "https://github.com/KankaveeRamsri/thai-dsr.git"
TIME_BUDGET_H = 11.0             # do not start a layer that would end after this (Kaggle stops at 12 h)
CHECK_REL_TOL = 1e-3             # layer-9 reference check: max relative L2 error
SMOKE = False                    # True: 36 clips, 2 epochs, outputs in runs_smoke/ (quick interactive test)

OUT = "/kaggle/working/runs_smoke" if SMOKE else "/kaggle/working/runs"
INPUT_ROOT = "/kaggle/input"
assert all(isinstance(l, int) and 0 <= l <= 24 for l in LAYERS) and len(set(LAYERS)) == len(LAYERS), LAYERS

In [ ]:
import os, sys, time, json, shutil, subprocess, platform
T0 = time.time()
import torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"],
                     capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU: Settings > Accelerator > GPU T4"
GPU_NAME = torch.cuda.get_device_name(0)
if "T4" not in GPU_NAME:
    print(f"WARNING: GPU is {GPU_NAME}, not a T4 (the W2 runs used a T4). Continuing; it is recorded.")
torch.backends.cuda.matmul.allow_tf32 = False   # full fp32 (no-op on T4)
torch.backends.cudnn.allow_tf32 = False

# Scratch space outside /kaggle/working (which is saved as output, max ~20 GB).
candidates = [d for d in ("/kaggle/temp", "/tmp") if os.path.isdir(d)]
WORK_BASE = max(candidates, key=lambda d: shutil.disk_usage(d).free)
WORK = f"{WORK_BASE}/thai-dsr-work"
os.makedirs(WORK, exist_ok=True)
free_gb = shutil.disk_usage(WORK).free / 1e9
print(f"GPU: {GPU_NAME} | torch {torch.__version__} | CUDA {torch.version.cuda} | python {platform.python_version()}")
print(f"scratch: {WORK} ({free_gb:.0f} GB free)")
assert free_gb > 12, "need > 12 GB of scratch disk"

In [ ]:
REPO_DIR = f"{WORK}/thai-dsr"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--quiet", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "--quiet", "origin"], check=True)
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--quiet", REF], check=True)
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"],
                        capture_output=True, text=True, check=True).stdout.strip()
if len(REF) == 40:
    assert COMMIT == REF, (COMMIT, REF)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("repo", REPO_DIR, "| ref", REF, "| commit", COMMIT)

# Everything the training path imports is preinstalled on Kaggle; install only what is missing,
# so the image's torch/numpy stack is left alone.
import importlib
missing = [pip for mod, pip in [("transformers", "transformers>=4.40.0"), ("librosa", "librosa>=0.10.1"),
                                ("soundfile", "soundfile>=0.12.1"), ("yaml", "pyyaml"), ("matplotlib", "matplotlib"),
                                ("pandas", "pandas"), ("tqdm", "tqdm"), ("torchaudio", "torchaudio")]
           if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
import numpy as np, pandas as pd, transformers, librosa
VERSIONS = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
            "transformers": transformers.__version__, "numpy": np.__version__, "librosa": librosa.__version__,
            "pandas": pd.__version__}
print(VERSIONS, "| installed:", missing or "nothing")

In [ ]:
# Locate the attached dataset and verify every input against its checksum.
import glob, hashlib, tarfile

def sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1 << 24), b""):
            digest.update(chunk)
    return digest.hexdigest()

markers = glob.glob(f"{INPUT_ROOT}/**/kaggle_dataset.json", recursive=True)
assert len(markers) == 1, f"attach the dataset thai-dsr-cv27-ablation (found {markers})"
DS = os.path.dirname(markers[0])
def find(name):  # a file anywhere in the dataset (uploads may flatten or keep folders)
    hits = glob.glob(f"{DS}/**/{name}", recursive=True)
    assert len(hits) == 1, (name, hits)
    return hits[0]
SUMS = dict(reversed(line.split()) for line in open(find("SHA256SUMS")) if line.strip())
print("dataset:", DS)
for name, digest in SUMS.items():
    hits = glob.glob(f"{DS}/**/{os.path.basename(name)}", recursive=True)
    if not hits and name.endswith(".tar"):
        continue  # unpacked by Kaggle on upload: checked file by file below
    assert len(hits) == 1 and sha256(hits[0]) == digest, f"checksum mismatch or missing: {name}"
print("top-level files match SHA256SUMS")

# The two archives: unpack to scratch, or use them as Kaggle unpacked them.
DATA = f"{WORK}/data"
os.makedirs(DATA, exist_ok=True)
roots = {}
for tar_name, inner in [("cv27_mel_tempo.tar", "mel_cache/tempo"), ("cv27_distorted_wavs.tar", "distorted/severe")]:
    hits = glob.glob(f"{DS}/**/{tar_name}", recursive=True)
    if hits:
        if not os.path.isdir(f"{DATA}/{inner}"):
            with tarfile.open(hits[0]) as tf:
                tf.extractall(DATA, **({"filter": "data"} if hasattr(tarfile, "data_filter") else {}))
        roots[tar_name] = DATA
    else:
        found = [p for p in glob.glob(f"{DS}/**/{inner}", recursive=True) if os.path.isdir(p)]
        assert len(found) == 1, f"{tar_name}: neither the archive nor an unpacked {inner}/ found"
        roots[tar_name] = found[0][: -len(inner)].rstrip("/")
MEL_ROOT, WAV_ROOT = roots["cv27_mel_tempo.tar"], roots["cv27_distorted_wavs.tar"]
start = time.time()
bad = []
for line in open(find("files.sha256")):
    digest, rel = line.split()
    root = MEL_ROOT if rel.startswith("mel_cache/") else WAV_ROOT
    if sha256(f"{root}/{rel}") != digest:
        bad.append(rel)
assert not bad, f"{len(bad)} file(s) differ, e.g. {bad[:3]}"
print(f"all files inside the archives verified in {time.time() - start:.0f}s | mel {MEL_ROOT} | wav {WAV_ROOT}")

for name in ["cv27_th_speaker_split.csv", "cv27_th_splits.json", "cv27_th_distortion_params.csv",
             "cv27_th_clean_16k_index.csv"]:
    assert sha256(find(name)) == sha256(f"data/manifests/{name}"), f"{name} differs from the repo at {COMMIT[:7]}"
params = pd.read_csv("data/manifests/cv27_th_distortion_params.csv", usecols=["distorted_path", "distorted_sha256"])
wav_sums = {rel: d for d, rel in (l.split() for l in open(find("files.sha256")))}
assert all(wav_sums[p] == h for p, h in zip(params["distorted_path"], params["distorted_sha256"])), \
    "distorted wavs differ from the frozen checksums tracked in the repo"
print("manifests equal the repo's; wavs equal the frozen set tracked in data/manifests/")

In [ ]:
# Training manifest (paths resolved against WAV_ROOT) and the fixed split, as on Colab.
import yaml
env = dict(os.environ, DATA_ROOT=WAV_ROOT)
subprocess.run([sys.executable, "scripts/build_cv_training_manifest.py", "--manifest", f"{WORK}/training_manifest.csv",
                "--splits", f"{WORK}/splits.json"], check=True, env=env)
SPLITS = json.load(open("data/manifests/cv27_th_splits.json"))["splits"]
assert json.load(open(f"{WORK}/splits.json"))["splits"] == SPLITS
manifest = pd.read_csv(f"{WORK}/training_manifest.csv")
severe = manifest[manifest["severity"] == "severe"].reset_index(drop=True)
assert len(severe) == 2996, len(severe)
MANIFEST = f"{WORK}/training_manifest.csv"
if SMOKE:  # 24 train / 6 val / 6 test clips
    keep = set(sorted(SPLITS["train"])[:24] + sorted(SPLITS["val"])[:6] + sorted(SPLITS["test"])[:6])
    severe = severe[severe["utterance_id"].isin(keep)].reset_index(drop=True)
    MANIFEST = f"{WORK}/training_manifest_smoke.csv"
    severe.to_csv(MANIFEST, index=False)
WAV_PATHS = list(severe["distorted_path"])
assert all(os.path.exists(p) for p in WAV_PATHS)

train_cfg = yaml.safe_load(open("configs/train.yaml"))
FIXED = {
    "severity": train_cfg["data"]["severity"], "batch_size": train_cfg["optim"]["batch_size"],
    "learning_rate": train_cfg["optim"]["learning_rate"], "optimizer": train_cfg["optim"]["optimizer"],
    "weight_decay": train_cfg["optim"]["weight_decay"], "lr_scheduler": train_cfg["optim"]["lr_scheduler"],
    "num_epochs": train_cfg["optim"]["num_epochs"], "seed": train_cfg["runtime"]["seed"],
    "input_norm": train_cfg["data"].get("input_norm"), "alignment": "tempo",
}
assert FIXED == {"severity": "severe", "batch_size": 4, "learning_rate": 0.001, "optimizer": "adam",
                 "weight_decay": 0.0, "lr_scheduler": "reduce_on_plateau", "num_epochs": 50, "seed": 42,
                 "input_norm": True, "alignment": "tempo"}, FIXED
print(f"{len(WAV_PATHS)} severe clips | recipe {FIXED}")

In [ ]:
# wav2vec2 + layer-9 reference check: stop before any training if extraction differs from W2.
from src.preprocessing.extract_embedding import extract_all_hidden_states, load_model, save_hidden_states
DEVICE = torch.device("cuda")
feature_extractor, w2v = load_model(DEVICE)
MODEL_REVISION = getattr(w2v.config, "_commit_hash", None)
assert w2v.config.num_hidden_layers == 24 and w2v.config.do_stable_layer_norm
reference = np.load(find("layer09_reference.npz"))
ref_meta = json.loads(str(reference["meta"]))
CHECK = []
for key in ref_meta["clips"]:
    hidden = extract_all_hidden_states(f"{WAV_ROOT}/{ref_meta['wav'][key]}", feature_extractor, w2v, DEVICE)
    assert len(hidden) == 25
    new, old = hidden[9].squeeze(0).cpu().numpy(), reference[key]
    assert new.shape == old.shape, (key, new.shape, old.shape)
    CHECK.append({"clip": key, "rel_l2": float(np.linalg.norm(new - old) / np.linalg.norm(old)),
                  "max_abs": float(np.abs(new - old).max())})
worst = max(c["rel_l2"] for c in CHECK)
print(f"model revision {MODEL_REVISION} | layer 9 vs W2 reference on {len(CHECK)} clips: "
      f"max relative L2 {worst:.2e}, max abs {max(c['max_abs'] for c in CHECK):.2e}")
if worst > CHECK_REL_TOL:
    raise RuntimeError(f"layer-9 extraction differs from the W2 archive by {worst:.2e} > {CHECK_REL_TOL}: stopping")

In [ ]:
import re

def extract_layer(layer, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    start = time.time()
    for i, path in enumerate(WAV_PATHS, 1):
        hidden = extract_all_hidden_states(path, feature_extractor, w2v, DEVICE)
        save_hidden_states(hidden, path, out_dir, layer=layer, verbose=False)
        if i % 500 == 0 or i == len(WAV_PATHS):
            print(f"  extract layer {layer}: {i}/{len(WAV_PATHS)} {time.time() - start:.0f}s", flush=True)
    n = len([f for f in os.listdir(out_dir) if f.endswith(f"_layer{layer:02d}.npy")])
    assert n == len(WAV_PATHS), (layer, n)
    return time.time() - start

def train_layer(layer, emb_dir, run_dir, tmp_dir):
    cmd = [sys.executable, "-u", "-m", "src.training.train",
           "--manifest", MANIFEST, "--splits", "data/manifests/cv27_th_splits.json", "--fixed-splits",
           "--embeddings-dir", emb_dir, "--mel-cache-dir", f"{MEL_ROOT}/mel_cache", "--preload", "--num-workers", "0",
           "--alignment", "tempo", "--distortion-params", "data/manifests/cv27_th_distortion_params.csv",
           "--layer", str(layer), "--device", "cuda",
           "--checkpoint-dir", f"{tmp_dir}/checkpoints", "--log-dir", f"{run_dir}/logs",
           "--best-checkpoint", f"{run_dir}/best.pt", "--loss-curve", f"{run_dir}/loss_curve.png",
           "--save-latest", f"{tmp_dir}/latest.pt"]
    if SMOKE:
        cmd += ["--num-epochs", "2"]
    env = dict(os.environ, TQDM_DISABLE="1")
    with open(f"{run_dir}/train.log", "a") as log:
        log.write(f"===== {time.strftime('%Y-%m-%d %H:%M:%S')} layer {layer} on {GPU_NAME} at {COMMIT}\n"
                  f"$ {' '.join(cmd)}\n")
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
        for line in process.stdout:
            log.write(line)
            log.flush()
            if line.startswith(("epoch ", "Input normalisation", "Training complete", "Held-out", "Traceback")) \
                    or "Error" in line:
                print(f"  [L{layer:02d}] {line}", end="", flush=True)
        return process.wait()

In [ ]:
os.makedirs(OUT, exist_ok=True)
SUMMARY = []
durations = []
for layer in LAYERS:
    run_dir = f"{OUT}/cv27_layer{layer:02d}_tempo"
    if os.path.exists(f"{run_dir}/DONE"):
        print(f"layer {layer}: already done, skipped")
        SUMMARY.append({"layer": layer, "status": "already done"})
        continue
    elapsed = time.time() - T0
    estimate = 1.15 * max(durations) if durations else 0
    if elapsed + estimate > TIME_BUDGET_H * 3600:
        print(f"layer {layer}: skipped, {elapsed / 3600:.1f} h used and the last layer took "
              f"{max(durations) / 60:.0f} min (budget {TIME_BUDGET_H} h)")
        SUMMARY.append({"layer": layer, "status": "skipped: time budget"})
        continue
    layer_start = time.time()
    emb_dir, tmp_dir = f"{WORK}/emb_layer{layer:02d}", f"{WORK}/tmp_layer{layer:02d}"
    shutil.rmtree(run_dir, ignore_errors=True)
    os.makedirs(run_dir)
    info = {"config": "tempo", "layer": layer, "input_norm": True, "commit": COMMIT, "ref": REF, "fixed": FIXED,
            "platform": "kaggle", "gpu": GPU_NAME, "versions": VERSIONS, "model_revision": MODEL_REVISION,
            "dataset_sha256": SUMS, "layer9_check": {"max_rel_l2": worst, "tolerance": CHECK_REL_TOL, "clips": CHECK},
            "smoke": SMOKE, "sessions": [{"started": time.strftime("%Y-%m-%d %H:%M:%S"), "gpu": GPU_NAME,
                                          "commit": COMMIT, "platform": "kaggle"}]}
    json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2)
    row = {"layer": layer}
    try:
        print(f"\n===== layer {layer} ({time.strftime('%H:%M:%S')}, {elapsed / 3600:.2f} h elapsed)", flush=True)
        row["extract_s"] = extract_layer(layer, emb_dir)
        train_start = time.time()
        code = train_layer(layer, emb_dir, run_dir, tmp_dir)
        row["train_s"] = time.time() - train_start
        if code != 0:
            raise RuntimeError(f"train.py exited with {code} (see {run_dir}/train.log)")
        log_text = open(f"{run_dir}/train.log").read()
        epoch_times = [float(t) for t in re.findall(r"^epoch \d+ time ([0-9.]+)s", log_text, re.M)]
        test = re.findall(r"^Held-out test loss: ([0-9.]+)", log_text, re.M)
        best = torch.load(f"{run_dir}/best.pt", map_location="cpu", weights_only=True)
        norm = best.get("input_norm") or {}
        assert norm.get("split") == "train" and norm.get("layer") == layer, norm
        assert SMOKE or norm.get("n_clips") == 2383, norm
        assert best["model_config"]["mapper"]["input_norm"] and os.path.exists(f"{run_dir}/input_norm.npz")
        row.update(status="ok", best_epoch=best.get("best_epoch"), best_val_loss=float(best["best_val_loss"]),
                   epochs=len(epoch_times), epoch_s=float(np.mean(epoch_times)) if epoch_times else None,
                   test_loss=float(test[-1]) if test else None)
        info["result"] = {**row, "input_norm": norm}
        json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2)
        open(f"{run_dir}/DONE", "w").close()
    except Exception as exc:  # keep going with the next layer; the summary shows the failure
        row.update(status=f"FAILED: {exc}")
        info["result"] = row
        json.dump(info, open(f"{run_dir}/run_info.json", "w"), indent=2)
        print(f"layer {layer} FAILED: {exc}", flush=True)
    finally:
        shutil.rmtree(emb_dir, ignore_errors=True)
        shutil.rmtree(tmp_dir, ignore_errors=True)
        torch.cuda.empty_cache()
    row["total_min"] = (time.time() - layer_start) / 60
    durations.append(time.time() - layer_start)
    SUMMARY.append(row)
    print(f"layer {layer}: {row['status']}, best epoch {row.get('best_epoch')}, best val loss "
          f"{row.get('best_val_loss')}, {row.get('epoch_s') or float('nan'):.1f} s/epoch, "
          f"{row['total_min']:.0f} min", flush=True)

In [ ]:
summary = pd.DataFrame(SUMMARY)
total_h = (time.time() - T0) / 3600
out_gb = sum(os.path.getsize(os.path.join(d, f)) for d, _, fs in os.walk("/kaggle/working") for f in fs) / 1e9
json.dump({"gpu": GPU_NAME, "commit": COMMIT, "hours": total_h, "output_gb": out_gb, "layer9_check_max_rel_l2": worst,
           "layers": SUMMARY}, open(f"{OUT}/summary.json", "w"), indent=2, default=str)
print(f"\n===== SUMMARY | GPU {GPU_NAME} | commit {COMMIT[:7]} | {total_h:.2f} h | output {out_gb:.2f} GB "
      f"| layer-9 check {worst:.1e}")
cols = [c for c in ["layer", "status", "best_epoch", "best_val_loss", "epochs", "epoch_s", "test_loss",
                    "extract_s", "train_s", "total_min"] if c in summary.columns]
print(summary[cols].to_string(index=False, float_format=lambda v: f"{v:.4g}"))
for d in sorted(os.listdir(OUT)):
    if os.path.isdir(f"{OUT}/{d}"):
        print(d, sorted(os.listdir(f"{OUT}/{d}")))
failed = [r["layer"] for r in SUMMARY if str(r.get("status", "")).startswith("FAILED")]
print("FAILED layers:", failed or "none")